# CP Development (mouse)

In [1]:
import numpy as np
np.random.default_rng(2159)

PYTHONHASHSEED=0

In [2]:
#load packages
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import os

sc.settings.verbosity = 3   
sc.set_figure_params(scanpy=True, fontsize=12, dpi_save=180, format='svg', transparent=True)
sc.settings.set_figure_params(dpi=80)

%matplotlib inline

image_format = 'svg' # e.g .png, .svg, etc.

In [3]:
my_c = ['#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4']

# Load data

In [1]:
#P2
P2_sample1 = sc.read_10x_h5('P2_sample1_filtered_feature_bc_matrix.h5')

#P3
P3_sample1 = sc.read_10x_h5('P3_sample1_filtered_feature_bc_matrix.h5')

P3_sample2 = sc.read_10x_h5('P3_sample2_filtered_feature_bc_matrix.h5')

#P11
P11_sample1 = sc.read_10x_h5('P11_sample1_filtered_feature_bc_matrix.h5')

P11_sample2 = sc.read_10x_h5('P11_sample2_filtered_feature_bc_matrix.h5')

#P60
P60_sample1 = sc.read_10x_h5('P60_sample1_filtered_feature_bc_matrix.h5')

P60_sample2 = sc.read_10x_h5('P60_sample2_filtered_feature_bc_matrix.h5')

P60_sample3 = sc.read_10x_h5('P60_sample3_filtered_feature_bc_matrix.h5')

In [ ]:
adata_dict = {'P2_sample1': P2_sample1,
              'P3_sample1': P3_sample1,
              'P3_sample2': P3_sample2, 
              'P11_sample1': P11_sample1,
              'P11_sample2': P11_sample2,
              'P60_sample1': P60_sample1,
              'P60_sample2': P60_sample2,
              'P60_sample3': P60_sample3}

In [6]:
for k, v in adata_dict.items():
    v.var_names_make_unique()
    v.obs_names_make_unique()
    print(f"{k} \n {v}")

In [7]:
i = 0
for k, v in adata_dict.items():
    if k.split('_')[-1] == 'P2':
        adata_dict[k].obs['Age'] = 'P2/P3'
        adata_dict[k].obs['Gender'] = 'Mix'
        adata_dict[k].obs['Project'] = f"{k.split('_')[0]}_{k.split('_')[1]}"
        
    elif k.split('_')[-1] == 'P3':
        adata_dict[k].obs['Age'] = 'P2/P3'
        adata_dict[k].obs['Gender'] = 'Mix'
        adata_dict[k].obs['Project'] = f"{k.split('_')[0]}_{k.split('_')[1]}"
        
    else:  
        adata_dict[k].obs['Age'] = k.split('_')[-1]
        adata_dict[k].obs['Gender'] = 'Mix'
        adata_dict[k].obs['Project'] = f"{k.split('_')[0]}_{k.split('_')[1]}"

In [8]:
for sample in adata_dict.keys():
    adata_dict[sample].obs['Age'] = adata_dict[sample].obs['Age'].astype('category')
    adata_dict[sample].obs['Gender'] = adata_dict[sample].obs['Gender'].astype('category')
    adata_dict[sample].obs['Project'] = adata_dict[sample].obs['Project'].astype('category')

In [9]:
i = 0
first = False
rest = []
categories = []

for k, v in adata_dict.items():
    if i==0:
        first = v
        i += 1
        categories.append(k)
    else:
        rest.append(v)
        categories.append(k)


adata = first.concatenate(rest, batch_categories=categories)

In [10]:
adata

In [11]:
adata.obs['batch']

In [12]:
adata.obs['Age']

# Add QC metrics

## Mitochondrial, ribossomal and hemoglobin genes

In [13]:
#add mito, ribo and hemo gene info
adata.var['Mito'] = adata.var_names.str.startswith('mt-') 
adata.var['Ribo'] = adata.var_names.str.startswith(("Rps","Rpl"))
adata.var['Hb'] = adata.var_names.str.contains(("^Hb[^(p)]"))

sc.pp.calculate_qc_metrics(adata, qc_vars=['Mito','Ribo','Hb'], percent_top=None, log1p=False, inplace=True)

In [14]:
#Sample
annot = sc.queries.biomart_annotations(
        "mmusculus",
        ["ensembl_gene_id", "external_gene_name", "start_position", "end_position", "chromosome_name"],
    ).set_index("external_gene_name")


## Sample sex

In [15]:
adata

In [16]:
#sample sex
chrY_genes = adata.var_names.intersection(annot.index[annot.chromosome_name == "Y"])
adata.obs['percent_chrY'] = np.sum(adata[:, chrY_genes].X, axis=1).A1 / np.sum(adata.X, axis=1).A1 * 100
adata.obs["Xist-counts"] = adata.X[:,adata.var_names.str.match('Xist')].toarray()

In [17]:
sc.pl.violin(adata, ["Xist-counts", "percent_chrY"], jitter=0, rotation= 90, palette=my_c)

## Calculate cell-cycle scores

In [ ]:

s_genes = [x.strip() for x in open('cell_cycle_S_geneList.txt')]
g2m_genes = [x.strip() for x in open('cell_cycle_G2M_geneList.txt')]

sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

## Saving raw data

In [ ]:
adata.write_h5ad('Development_unfiltered.h5ad')

In [ ]:
adata = sc.read_h5ad('Development_unfiltered.h5ad')

In [32]:
adata

In [22]:
adata

## Ploting data before QC

In [6]:
sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'],
             ncols=2, rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_rawdata_all.svg')

In [24]:
sc.pl.violin(adata, ["Xist-counts", "percent_chrY"], jitter=0, groupby = 'batch', rotation= 90, palette=my_c)

In [25]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)

In [26]:
%matplotlib inline


sc.pl.umap(adata, color=['Gender', 'Xist-counts'], vmax=1, palette=my_c)

In [27]:
%matplotlib inline


sc.pl.umap(adata, color=['Gender', 'percent_chrY'], vmax=0.1)

In [ ]:
fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(13,12))
sns.countplot(y = 'batch',data = adata.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')
    
plt.show()

image_name = 'barplots_rawdata_all.svg'

fig.savefig(image_name, format=image_format, dpi=180)

In [29]:
sc.pl.highest_expr_genes(adata,n_top=20)

In [30]:
sc.pl.violin(adata, ['S_score', 'G2M_score'], jitter=0, groupby = 'batch', rotation=90, palette=my_c)

In [31]:
adata

# QC

## Data normalization and logaritmization

In [33]:
## normalize to depth 10 000
sc.pp.normalize_per_cell(adata,counts_per_cell_after=1e4)

## logaritmize
sc.pp.log1p(adata)

**Creating raw data object**

In [34]:
adata.raw = adata.copy()

## Searching for doublets

In [9]:
import scrublet as scr

In [10]:
##  Predict doublets
import scrublet as scr

scrub = scr.Scrublet(adata.raw.X)
doublet_score, predicted_doublets = scrub.scrub_doublets()

In [11]:
%matplotlib inline

In [12]:
scrub.plot_histogram()

In [13]:
predicted_doublets = scrub.call_doublets(threshold=0.33)

In [14]:
scrub.plot_histogram()

In [15]:
adata.obs['predicted_doublets'] = predicted_doublets

In [16]:
sum(adata.obs['predicted_doublets'])

In [17]:
## add in column with singlet/doublet instead of True/False
adata.obs['doublet_info'] = adata.obs["predicted_doublets"].astype(str)
sc.pl.violin(adata, 'n_genes_by_counts', jitter=0.4, groupby = 'doublet_info', rotation=45, palette=my_c, save='_doublets_all.svg')

In [35]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)

In [19]:
%matplotlib inline


sc.pl.umap(adata, color=['doublet_info'], palette=my_c, save='_doublets_all.svg')

In [20]:
adata = adata[adata.obs['doublet_info'] == 'False',:]
print(adata.shape)

In [21]:
adata

## Saving data without doublets

In [ ]:
adata.write_h5ad('Development_noDBL.h5ad')

In [ ]:
adata = sc.read_h5ad('Development_noDBL.h5ad')

In [23]:
adata

## Filtering cells and genes

In [24]:
sc.pp.filter_cells(adata,min_genes = 250)
sc.pp.filter_genes(adata,min_cells = 5)

In [25]:
%matplotlib inline

sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_prefilter_all.svg')

In [26]:
adata = adata[adata.obs['total_counts'] < 80000,:] 

In [27]:
adata = adata[adata.obs['n_genes_by_counts'] < 9000, :]

In [28]:
adata = adata[adata.obs['pct_counts_Mito'] < 20,:]

In [29]:
adata = adata[adata.obs['pct_counts_Hb'] < 0.05,:]

In [30]:
adata = adata[adata.obs['pct_counts_Ribo'] < 20,:]

In [31]:
adata = adata[adata.obs['pct_counts_Ribo'] > 2.5,:]

In [32]:
%matplotlib inline

sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='batch', palette=my_c, save='_afterfilter_all.svg')

In [33]:
print("Remaining cells %d" % adata.n_obs)

In [34]:
adata

In [ ]:

fig, (ax1,ax2) = plt.subplots(nrows=2,ncols=1,constrained_layout=True,figsize=(13,12))
sns.countplot(y = 'batch',data = adata.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)
    ax1.set_ylabel('Cells in each sample')

sns.countplot(y = 'Age',data = adata.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)
    ax2.set_ylabel('Cells in each time point')
    
plt.show()

image_name = 'barplots_afterfilter_all.svg'

fig.savefig(image_name, format=image_format, dpi=180)

## Saving filtered data

In [ ]:
adata.write_h5ad('Development_noDBL_filtered.h5ad')

## Loading filtered data

In [ ]:
adata = sc.read_h5ad('Development_noDBL_filtered.h5ad')

In [37]:
adata

In [10]:
adata

# Dimensionality redutcion

## HVG

In [40]:
#HVGs
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5, flavor='seurat_v3', n_top_genes=4000)
print("Highly variable genes: %d"%sum(adata.var.highly_variable))

In [41]:
sc.pl.highly_variable_genes(adata)

In [42]:
var_genes_all = adata.var.highly_variable

In [43]:
adata = adata[:, var_genes_all]

In [44]:
adata

In [45]:
sc.pp.scale(adata,max_value=10)

## PCA

In [46]:
sc.tl.pca(adata, svd_solver='arpack')

In [47]:
sc.pl.pca(adata, color='batch', components = ['1,2','3,4','5,6','7,8'], ncols=2, wspace=0.8, palette=my_c)

In [48]:
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, save='n_pcs_all.svg')

# Clustering

In [83]:
sc.pp.neighbors(adata, n_pcs=32, n_neighbors=15)
sc.tl.umap(adata)

In [84]:
sc.pl.umap(adata, color=["batch", "Age"], size=1, wspace=0.8, ncols=2, palette=my_c, save='pre_correction_all.svg')

## Batch correction

### Harmony

In [4]:
from harmony import harmonize

In [72]:
Z = harmonize(adata.obsm['X_pca'], adata.obs, batch_key = 'batch')
adata.obsm['X_harmony'] = Z

In [85]:
print("Calculating UMAP")
sc.pp.neighbors(adata, n_pcs=32, n_neighbors=15, use_rep="X_harmony")
sc.tl.umap(adata)

In [92]:
print("Ploting UMAP")
sc.pl.umap(adata, color=["batch", 'Age'], size=1.5, wspace=0.8, ncols=2, palette=my_c, save='after_correction_all.svg')

## Clustering

In [87]:
# clustering
for res in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]:
    sc.tl.leiden(adata, resolution=res,key_added = 'leiden_%s'%res)

In [93]:
#plot
sc.pl.umap(adata,color=['leiden_0.1', 'leiden_0.2', 'leiden_0.3', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 
                        'leiden_0.8', 'leiden_1.0'], ncols=4, wspace=1, cmap='tab20')

In [162]:
%matplotlib inline

sc.pl.violin(adata, keys=['total_counts','n_genes_by_counts', 'pct_counts_Mito', 'pct_counts_Hb', 'pct_counts_Ribo'], 
             rotation=90,stripplot=False,groupby='leiden_0.8')

## Saving clsutered data

In [ ]:
adata.write_h5ad('Development_noDBL_filtered_clustered.h5ad')

## Loading filtered data

In [ ]:
adata = sc.read_h5ad('Development_noDBL_filtered_clustered.h5ad')

In [91]:
adata

In [31]:
adata

### Leiden_0.1

In [41]:
sc.tl.rank_genes_groups(adata, 'leiden_0.1', method='wilcoxon', key_added = "wilcoxon_01", use_raw=True)

In [95]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_01" , groupby="leiden_0.1", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.2

In [43]:
sc.tl.rank_genes_groups(adata, 'leiden_0.2', method='wilcoxon', key_added = "wilcoxon_02", use_raw=True)

In [96]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_02" , groupby="leiden_0.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.3

In [45]:
sc.tl.rank_genes_groups(adata, 'leiden_0.3', method='wilcoxon', key_added = "wilcoxon_03", use_raw=True)

In [97]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_03" , groupby="leiden_0.3", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.4

In [47]:
sc.tl.rank_genes_groups(adata, 'leiden_0.4', method='wilcoxon', key_added = "wilcoxon_04", use_raw=True)

In [98]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_04" , groupby="leiden_0.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.5

In [49]:
sc.tl.rank_genes_groups(adata, 'leiden_0.5', method='wilcoxon', key_added = "wilcoxon_05", use_raw=True)

In [99]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_05" , groupby="leiden_0.5", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.6

In [52]:
sc.tl.rank_genes_groups(adata, 'leiden_0.6', method='wilcoxon', key_added = "wilcoxon_06", use_raw=True)

In [100]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_06" , groupby="leiden_0.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_0.8

In [55]:
sc.tl.rank_genes_groups(adata, 'leiden_0.8', method='wilcoxon', key_added = "wilcoxon_08", use_raw=True)

In [101]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_08" , groupby="leiden_0.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.0

In [58]:
sc.tl.rank_genes_groups(adata, 'leiden_1.0', method='wilcoxon', key_added = "wilcoxon_1", use_raw=True)

In [102]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_1" , groupby="leiden_1.0", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.2

In [61]:
sc.tl.rank_genes_groups(adata, 'leiden_1.2', method='wilcoxon', key_added = "wilcoxon_12", use_raw=True)

In [103]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_12" , groupby="leiden_1.2", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.4

In [64]:
sc.tl.rank_genes_groups(adata, 'leiden_1.4', method='wilcoxon', key_added = "wilcoxon_14", use_raw=True)

In [104]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_14" , groupby="leiden_1.4", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.6

In [67]:
sc.tl.rank_genes_groups(adata, 'leiden_1.6', method='wilcoxon', key_added = "wilcoxon_16", use_raw=True)

In [105]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_16" , groupby="leiden_1.6", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_1.8

In [70]:
sc.tl.rank_genes_groups(adata, 'leiden_1.8', method='wilcoxon', key_added = "wilcoxon_18", use_raw=True)

In [106]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_18" , groupby="leiden_1.8", cmap="YlGnBu", vmin=0, vmax=10)

### Leiden_2.0

In [73]:
sc.tl.rank_genes_groups(adata, 'leiden_2.0', method='wilcoxon', key_added = "wilcoxon_2", use_raw=True)

In [107]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_2" , groupby="leiden_2.0", cmap="YlGnBu", vmin=0, vmax=10)

## Saving clsutered data

In [ ]:
adata.write_h5ad('Development_noDBL_filtered_clustered.h5ad')

## Loading clustered data

In [ ]:
adata = sc.read_h5ad('Development_noDBL_filtered_clustered.h5ad')

In [71]:
adata

In [78]:
adata

In [ ]:
adata_all = sc.read_h5ad('Development_unfiltered.h5ad')

In [110]:
markers = {"Epithelial cells": ['Gata3','Atp10b','Cidea', "Krt18","Htr2c","Car12","Clic6", "Folr1","Igfbp2",'Otx2','Chchd10','Clu','Calml4'],
           "Mesenchymal cells": ['Pdgfrb','Mgp','Cfh','Dcn','Tgfbi','Moxd1','Cp','Bgn','Cxcl12','Col3a1'],
           "Endothelial cells": ["Egfl7","Kdr","Pecam1", "Cd34", "Plvap",'Cdh5','Flt1','Igfbp3','Ly6a','Ly6c1'],
           "Immune cells": ["C1qa",'Tyrobp','Fcer1g','Ctss','C1qb','Ptprc','C1qc','Cd83','Lyz2','Laptm5'],
           "NeuroGlia": ['Tubb2b','Gpm6b','Rtn1','Olig1','Fabp7','Bcan','Cnp','Scrg1','Opcml','Pcsk1n'], 
           "NIPC": ["Cdca7", "2810417H13Rik", "Rrm2", "Ube2c", "Top2a", "Birc5", "P2rx3"]}

In [111]:
for k, v in markers.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=6)

In [ ]:
markers2 = {'Ciliogenesis': ['Shisa8', 'Ccdc67', 'Ccno', 'Mcidas'], 
                'Mesenchymal cells': ['Dcn', 'Adam12', 'Cxcl12', 'Lum'], 
                'Epithelial cells': ['Htr2c', 'Enpp2', 'Kcnj13', 'Aqp1', 'Gata3'], 
                'Endothelial cells': ['Plvap', 'Flt1', 'Pecam1'],
                'Neuro / Glial': ['Slc1a3', 'Olig1', 'Olig2', 'Dlx1', 'Dlx2', 'Eomes', 'Tbr2'],
                'Macrophages': ['Cx3cr1', 'Mrc1', 'C1qc', 'Cfsr1'], 
                'Lymphocytes': ['Cd3g', 'Cxcr6', 'Txk', 'Tcf7', 'Tnfrsf25'], 
                'Monocytes': ['Cd14', 'Cd44', 'Ccr2', 'Wfdc17', 'Ms4a4c', 'Sirpb1c'], 
                'Mast cells': ['Kit', 'Smpx', 'Clec12b'], 
                'B cells': ['Cd79a', 'Cd19', 'Vpreb3', 'Il7r'], 
                'Basophils': ['Gata2', 'Fcer1a2', 'Cd200r3', 'Csf1'],
                'Neutrophils': ['S110a9', 'Ly6g', 'Ly6ca'], 
                'DC': ['Cd74', 'H2.Ab1', 'H2.Aa'], 
                'Undefined': ['Gadd45b', 'Ddit3', 'Htra2', 'Nr1d1'], 
                'EMT': ['Cdh2', 'Cdh11', 'Itga5', 'Itgb6', 'Sdc1', 'Fsp1', 'Vim', 'Ctnnb1', 'Fn1', 'Lama5', 'Col1a1', 'Snail1', 'Zeb1', 'Foxc2', 'Gsc']} 

In [ ]:
for k, v in markers2.items():
    genes_in_data = []
    for gene in v:
        if gene in adata.raw.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=5)

In [ ]:
marker3 = {'Neuroblasts': ['Tubb3', 'Dcx', 'Islr2'], 
            'TAPs': ['Mki67', 'Ube2c', 'Top2a'], 
            'NSCs': ['Helt', 'Cdk6', 'Mcm2'],
            'OPCs': ['Pdgfra', 'Cdo1', 'Lhfpl3', 'Snap25'], 
            'Neurons': ['Pcp4', 'Meg3', 'Nrgn', 'Snap25'],
            'Ependymal cells': ['Enkur', 'Foxj1', 'Ccdc153'],
            'Astrocytes': ['Slc1a3', 'S1pr1', 'Cldn10'], 
            'Microglia, activated': ['Ccl4', 'Egr1', 'Fos', 'Fosb'],
            'Endothelial cells': ['Flt1', 'Egfl7', 'Cldn5'],
            'Oligodendrocytes': ['Enpp2', 'Mog', 'Opalin', 'Sox10', 'Plp1', 'Mog', 'Mag', 'Car2'],
            'transition cells': ['Ascl1', 'Egfr', 'Notum', 'S100a6'],
            'COP': ['Gpr17', 'Tnr', 'Fyn']}

In [ ]:
for k, v in marker3.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=8)

In [116]:
adata

# Cluster annotation

**Select genes that belong to immune cells but are clustered into another cluster**

In [72]:
gene1 = 'Cd3g'
gene2 = 'Txk'
gene3 = 'Cxcr6'
gene4 = 'Cd79a'
gene5 = 'Cd19'
gene6 = 'Vpreb3'
gene7 = 'Il7r'
gene8 = 'Cd7'
gene9 = 'Ctsw'

adata.obs['CoEx'] = ((adata.raw[:,'{}'.format(gene1)].X.todense() > 0) |
                    (adata.raw[:,'{}'.format(gene2)].X.todense() > 0) |
                    (adata.raw[:,'{}'.format(gene3)].X.todense() > 0) | 
                    (adata.raw[:,'{}'.format(gene4)].X.todense() > 0) | 
                    (adata.raw[:,'{}'.format(gene5)].X.todense() > 0) | 
                    (adata.raw[:,'{}'.format(gene6)].X.todense() > 0) |
                    (adata.raw[:,'{}'.format(gene7)].X.todense() > 0) |
                    (adata.raw[:,'{}'.format(gene8)].X.todense() > 0) |
                    (adata.raw[:,'{}'.format(gene9)].X.todense() > 0))

In [73]:
cells = []

for cell in adata.obs_names:
    if adata.obs['CoEx'].loc[cell] == True:
        
        cells.append(cell)

In [74]:
adata.obs['CoEx'] = adata.obs['CoEx'].astype('category')

In [76]:
sc.pl.umap(adata, color=['CoEx', 'Txk', 'Cd3d', 'Cxcr6', 'Cd79a', 'Cd19', 'Vpreb3', 'Il7r', 'Cd7', 'Ctsw'], 
           palette=['white', 'Black'], cmap='YlGnBu')

In [ ]:
adata.obs['lymphocytes'] = False

In [ ]:
for cell in cells:
    if adata.obs['leiden_1.0'].loc[cell] == '21':
        adata.obs['leiden_1.0'].loc[cell] = '25'
        adata.obs['lymphocytes'].loc[cell] = True

In [18]:
adata.obs["Cell_type"] = adata.obs["leiden_1.0"].replace({"9": "Epithelial cells",
                                                          "6": "Epithelial cells", 
                                                          "3": "Epithelial cells",
                                                          "4": "Epithelial cells",
                                                          "7": "Epithelial cells",
                                                          "26": "Epithelial cells",
                                                          "2": "Epithelial cells", 
                                                          "0": "Epithelial cells",
                                                          "5": "Epithelial cells",
                                                          "1": "Epithelial cells",
                                                          "8": "Epithelial cells",
                                                          "13": "Endothelial cells",
                                                          "11": "Endothelial cells",
                                                          "22": "Endothelial cells",
                                                          "16": "Neuronal and glial cells",
                                                          "17": "Epithelial cells",
                                                          "19": "Epithelial cells",
                                                          "21": "Epithelial cells",
                                                          "24": "Neuronal and glial cells", 
                                                          "18": "Epithelial cells", 
                                                          "23": "Neuronal and glial cells",
                                                          "15": "Mesenchymal cells", 
                                                          "10": "Mesenchymal cells", 
                                                          "14": "Mesenchymal cells", 
                                                          "20": "Immune cells", 
                                                          "12": "Immune cells", 
                                                          "25": "Immune cells"}) 


In [20]:
sc.tl.rank_genes_groups(adata, 'Cell_type', method='wilcoxon', key_added = "wilcoxon_Cell_type", use_raw=True)

In [21]:
adata.obs['Cell_type'].unique()

In [22]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_Cell_type" , groupby="Cell_type", cmap="YlGnBu", vmin=0, vmax=10,
                               save='_nofiltered_all.svg')

In [ ]:
sc.pl.umap(adata, color=['Cell_type', 'Age'], size=1, wspace=0.7, palette=my_c)

In [29]:
adata.obs['P2/P3'] = False
adata.obs['P11'] = False
adata.obs['P60'] = False

In [30]:
ages = ['P2/P3', 'P11', 'P60']

In [31]:
for p in ages:
    for cell in adata.obs_names:
        if adata.obs['Age'].loc[cell] == p:
            adata.obs[p].loc[cell] = p
        else:
            adata.obs[p].loc[cell] = np.nan

In [ ]:
sc.pl.umap(adata, color=ages, size=1, wspace=0.5, na_color="#FF000000", save='_ages_all.svg')

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="Cell_type", method='wilcoxon', n_genes=50)

clust_df = pd.DataFrame(adata.uns['rank_genes_groups']['names'])
clust_df.to_csv("Top_50_Postnatal_mouse_marker_genes_CP.csv")

In [40]:
markers1 = {"Epithelial cells": ['Folr1', 'Clic6', 'Otx2', 'Krt18'],
           "Neuron and Glia cells": ['Sox6', 'Ptprz1', 'Mbp', 'Plp1'],
           "Immune cells": ['C1qa', 'C1qb', 'Fcer1g', 'C1qc'],
           "Mesenchymal cells": ['Dcn', 'Itga8', 'Col3a1', 'Lum'],
           "Endothelial cells": ['Flt1', 'Plvap', 'Pecam1', 'Ly6a'],}

In [41]:
mp = sc.pl.matrixplot(adata, markers1, 'Cell_type', dendrogram=True,
                 colorbar_title='mean z-score', vmin=0, vmax=4, cmap='YlGnBu', return_fig=True)
mp.add_totals(color='turquoise').show()

mp.savefig('figures/matrixplot_clusters_all.svg', format='svg', dpi=180)

In [42]:
sc.pl.rank_genes_groups_stacked_violin(adata, var_names=markers1,cmap='YlGnBu',groupby='Cell_type',key='wilcoxon_Cell_type',
                                      figsize=(12,4), vmax=4, save='_clusters_all.svg')

In [43]:
for k, v in markers1.items():
    genes_in_data = []
    for gene in v:
        if gene in adata_all.var_names:
            genes_in_data.append(gene)
    print(f"{k} {'-'*90}")
    sc.pl.umap(adata, color=genes_in_data, use_raw=True, ncols=4, cmap='YlGnBu', vmax=6, save=f"_{k}_marker_genes_all.svg")

In [44]:
sc.pl.violin(adata, ['total_counts','n_genes_by_counts','pct_counts_Mito','pct_counts_Ribo','pct_counts_Hb'], 
             groupby="Cell_type", rotation=90, stripplot=False, palette=my_c, save='_celltypes_metrics.svg')

In [ ]:
fig, (ax1,ax2,ax3) = plt.subplots(nrows=3,ncols=1,constrained_layout=True,figsize=(16,12))
sns.countplot(y = 'batch',data = adata.obs,ax=ax1, palette=my_c)
for container in ax1.containers:
    ax1.bar_label(container)

sns.countplot(y = 'Cell_type',data = adata.obs,ax=ax2, palette=my_c)
for container in ax2.containers:
    ax2.bar_label(container)

sns.countplot(y = 'Age',data = adata.obs,ax=ax3, palette=my_c)
for container in ax3.containers:
    ax3.bar_label(container)

fig.savefig('barplots_celltypes_all.svg', format='svg')

In [46]:
def batch_pct_cellType(adata_file,groupby_key,legend_key,pos):
    tmp = pd.crosstab(adata_file.obs[groupby_key],adata_file.obs[legend_key], normalize='index')   
    tmp.plot.bar(stacked=True,rot= 90, grid=False, color=my_c,
                 xlabel='',ax=pos).legend(bbox_to_anchor=[1.20,1])

In [ ]:
fig, (ax1,ax2,ax3) = plt.subplots(3, 1, constrained_layout=True,figsize=(10,15))
batch_pct_cellType(adata,'Cell_type','Age',ax1)
batch_pct_cellType(adata,'Cell_type','batch',ax2)
batch_pct_cellType(adata,'Age','Cell_type',ax3)

fig.savefig('barplots_celltypes_age_all.svg', format='svg')

In [48]:
ct = pd.crosstab(adata.obs['batch'],adata.obs['Cell_type'], normalize='index')

In [49]:
ct

In [50]:
ct['Age'] = None

In [51]:
for b in ct.index:
    if 'P2' in b:
        ct['Age'].loc[b] = 'P2/P3'  
    elif 'P3' in b:
        ct['Age'].loc[b] = 'P2/P3'
    elif 'P11' in b:
        ct['Age'].loc[b] = 'P11'
    else:# 'P60' in b:
        ct['Age'].loc[b] = 'P60'

In [54]:
col1 = ct.columns.to_list()

In [58]:
ctt = ct.reset_index().melt(id_vars=['batch'])[:-8]

In [59]:
ctt.rename({'value': 'Percentage of cells'}, axis=1, inplace=True)

In [60]:
ctt['Age'] = None

In [61]:
ctt = ctt.set_index(['batch'])

In [62]:
ctt.index

In [63]:
for b in ctt.index:
    if 'P2' in b:
        ctt['Age'].loc[b] = 'P2/P3'  
    elif 'P3' in b:
        ctt['Age'].loc[b] = 'P2/P3'
    elif 'P11' in b:
        ctt['Age'].loc[b] = 'P11'
    else:# 'P60' in b:
        ctt['Age'].loc[b] = 'P60'

In [ ]:
from matplotlib.legend_handler import HandlerTuple

fig = plt.figure(figsize=(22,10))
ax = fig.add_subplot(111)    # The big subplot
ax1 = fig.add_subplot(211)
ax2 = fig.add_subplot(212)

# Turn off axis lines and ticks of the big subplot
ax.spines['top'].set_color('none')
ax.spines['bottom'].set_color('none')
ax.spines['left'].set_color('none')
ax.spines['right'].set_color('none')
ax.tick_params(labelcolor='w', top=False, bottom=False, left=False, right=False)
ax.grid(False)

sns.boxplot(x = ctt['Cell_type'], y = ctt['Percentage of cells'], data = ctt, hue=ctt['Age'], 
            palette = my_c, showfliers = True, ax=ax1)

sns.stripplot(data=ctt, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], ax=ax1)

sns.boxplot(x = ctt['Cell_type'], y = ctt['Percentage of cells'], data = ctt, hue=ctt['Age'], 
            palette = my_c, showfliers = True, ax=ax2)

sns.stripplot(data=ctt, x='Cell_type', y="Percentage of cells", hue="Age", 
              dodge=True, palette=my_c[6:9], ax=ax2)


ax1.set_ylim(0.70,1)   # those limits are fake
ax2.set_ylim(0,0.3)
sns.despine(ax=ax2)
sns.despine(ax=ax1, bottom=True)

d = .005  # how big to make the diagonal lines in axes coordinates
kwargs = dict(transform=ax1.transAxes, color='k', clip_on=False)
ax1.plot((-d, +d), (-d, +d), **kwargs)        # top-left diagonal

kwargs.update(transform=ax2.transAxes)  # switch to the bottom axes
ax2.plot((-d, +d), (1 - d, 1 + d), **kwargs)  # bottom-left diagonal

ax2.legend_.remove()
ax1.set(xlabel=None, ylabel=None)
ax2.set(xlabel=None,ylabel=None)
# ax1.legend_.remove()
ax1.set_xticks([])
ax1.set_yticks(np.arange(0.7,1, 0.1))
ax2.set_yticks(np.arange(0,0.31, 0.1))

from textwrap import wrap
names = [cell for cell in ctt['Cell_type'].unique()]

names = [ '\n'.join(wrap(l, 20)) for l in names ]

handles, labels = ax1.get_legend_handles_labels()
ax1.legend(handles=[(handles[0], handles[3]), (handles[1], handles[4]), (handles[2], handles[5])],
          labels=['P2/P3', 'P11', 'P60'], handlelength=2,
          handler_map={tuple: HandlerTuple(ndivide=None)},bbox_to_anchor=[1.12,1.05],fontsize=20)

ax2.set_xticklabels(names, fontsize=22)
# ax2.tick_params(names, labelsize=18)
ax1.tick_params(axis='y', labelsize=20)
ax2.tick_params(axis='y', labelsize=20)
ax.set_ylabel('Percantage of cells', fontsize=22)
ax.yaxis.set_label_coords(-0.05, 0.5)

plt
plt.show()

plt.savefig('boxplots_all_celltypes_all.svg')

## Saving annotated data

In [ ]:
adata.write_h5ad('Development_noDBL_filtered_clustered_annotated.h5ad')

## Loading annotated data

In [ ]:
adata = sc.read_h5ad('Development_noDBL_filtered_clustered_annotated.h5ad')

In [ ]:
adata

In [73]:
sc.set_figure_params(scanpy=True, fontsize=14)

In [ ]:
sc.pl.umap(adata, color=['Cell_type', 'Age'], ncols=4, cmap='magma',wspace=0.7, save='_celltypes_age_all.svg')

In [ ]:
sc.tl.filter_rank_genes_groups(adata, groupby='Cell_type', key = "wilcoxon_Cell_type", 
                               key_added='wilcoxon_filtered_leiden_Cell_type',
                               min_in_group_fraction=0.1,max_out_group_fraction=0.5,min_fold_change=0.25)

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=15, key="wilcoxon_filtered_leiden_Cell_type" , groupby="Cell_type", cmap="YlGnBu", vmin=0, vmax=5, save='_filtered_15_all.svg')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, key="wilcoxon_filtered_leiden_Cell_type" , groupby="Cell_type", cmap="YlGnBu", vmin=0, vmax=5, save='_filtered_10_all.svg')